In [0]:
%pip install openpyxl

In [0]:
from pathlib import Path
import numpy as np
import pandas as pd

FILE = Path("/Workspace/Users/weimings@andrew.cmu.edu/Datathon-2026/datathon_195_labels_en.xlsx")

# 1. 读取 Excel（只保留名单、查询经纬度和人工评分，旧特征仅留作对比）
labels = pd.read_excel(FILE, sheet_name="manual_labels", header=2, engine="openpyxl")
features = pd.read_excel(FILE, sheet_name="training_features", header=2, engine="openpyxl")

labels = labels[["row_id", "label"]].dropna(subset=["row_id"]).copy()
features = features.dropna(subset=["row_id"]).copy()

assert set(labels["row_id"]) == set(features["row_id"]), "Row IDs do not match"
base = features.merge(labels, on="row_id", validate="one_to_one")
base["label"] = pd.to_numeric(base["label"], errors="raise")

# 2. 统一服务名称：hemodialysis -> dialysis
base["service"] = base["service_code"].replace({"hemodialysis": "dialysis"})

# 备份旧特征，方便下一步检查哪些行证据变了
old_snapshot = base[["row_id", "evidence_score", "n_direct", "distance_km", "location_weight"]].rename(
    columns={
        "evidence_score": "old_evidence_score",
        "n_direct": "old_n_direct",
        "distance_km": "old_distance_km",
        "location_weight": "old_location_weight",
    }
)

# 3. 从 Spark 最新表拉取证据特征 (按 unique_id + service)
ev_cols = ["unique_id", "service", "evidence_score", "evidence_level", "n_direct", "n_supporting", "quotes"]
ev_df = (
    spark.table("workspace.datathon.facility_service_evidence")
    .select(*ev_cols)
    .toPandas()
    .drop_duplicates(subset=["unique_id", "service"])
)

# 4. 从 Spark 最新表拉取机构特征 (按 unique_id)
fac_cols = [
    "unique_id", "facility_type", "n_quality_flags", "flag_capability_bloat",
    "completeness", "location_confidence", "latitude", "longitude"
]
fac_df = (
    spark.table("workspace.datathon.facilities_clean")
    .select(*fac_cols)
    .toPandas()
    .drop_duplicates(subset=["unique_id"])
)

# 5. 连接新特征（保留原来的 195 行）
training_data = (
    base[["row_id", "query_id", "facility_id", "service", "query_lat", "query_lon", "label", "location_weight"]]
    .rename(columns={"facility_id": "unique_id", "location_weight": "old_location_weight"})
    .merge(ev_df, on=["unique_id", "service"], how="left", validate="many_to_one")
    .merge(fac_df, on="unique_id", how="left", validate="many_to_one")
)

# 6. 用最新经纬度重新计算直线距离 (Haversine km)
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * r * np.arcsin(np.sqrt(a))

training_data["distance_km"] = haversine_km(
    training_data["query_lat"], training_data["query_lon"],
    training_data["latitude"], training_data["longitude"]
)

# 映射 location_confidence -> location_weight（如未匹配则回退旧值）
loc_map = {"high": 0.9, "medium": 0.7, "low": 0.4, "unknown": 0.5}
mapped_w = training_data["location_confidence"].astype(str).str.lower().map(loc_map)
training_data["location_weight"] = mapped_w.fillna(training_data["old_location_weight"])

# 7. 重新计算模型衍生特征
training_data["log_distance"] = np.log1p(training_data["distance_km"])
training_data["log_direct"] = np.log1p(training_data["n_direct"])
training_data["is_hospital"] = training_data["facility_type"].astype(str).str.lower().eq("hospital").astype(int)
training_data["issue_count"] = pd.to_numeric(training_data["n_quality_flags"], errors="coerce")
training_data["capability_bloat"] = training_data["flag_capability_bloat"].astype(int)
training_data["facility_id"] = training_data["unique_id"]

# 8. 必须检查（队友要求的全部 assert）
assert len(training_data) == 195, f"Expected 195 rows, got {len(training_data)}"
assert training_data["row_id"].nunique() == 195, "Duplicate row_id found"
assert training_data["label"].notna().all(), "Missing label"
assert training_data["label"].isin([0, 1, 2, 3]).all(), "Invalid label"
assert training_data["evidence_score"].notna().all(), "Missing evidence_score after join!"
assert training_data["query_id"].nunique() == 15
assert training_data.groupby("query_id").size().eq(13).all()

# 赋值给 df，直接兼容后面的 Cell 3 和 Cell 4
df = training_data.copy()

print(f"Ready (refreshed from Spark): {len(df)} rows, {df['query_id'].nunique()} queries")
print(df["label"].value_counts().sort_index())

In [0]:
check = df[["row_id", "query_id", "service", "label", "evidence_score", "n_direct", "quotes"]].merge(
    old_snapshot, on="row_id"
)
changed = check[
    (check["evidence_score"] - check["old_evidence_score"]).abs() > 0.05
].sort_values("row_id")

print(f"证据分数变化 > 0.05 的行数: {len(changed)} / 195")
display(changed[["row_id", "service", "label", "old_evidence_score", "evidence_score", "old_n_direct", "n_direct", "quotes"]])

In [0]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import ndcg_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Use the eight features chosen before evaluating the model.
df["log_distance"] = np.log1p(df["distance_km"])
df["log_direct"] = np.log1p(df["n_direct"])

feature_cols = [
    "log_distance",
    "evidence_score",
    "log_direct",
    "location_weight",
    "is_hospital",
    "issue_count",
    "completeness",
    "capability_bloat",
]

X = df[feature_cols].apply(pd.to_numeric, errors="raise")
y = df["label"].astype(int)
groups = df["query_id"]

def new_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        Ridge(alpha=1.0),
    )

# Current 04_ranking formula for the three routine services in the sample.
df["baseline_score"] = (
    df["evidence_score"]
    * df["location_weight"]
    * np.exp(-df["distance_km"] / 75.0)
)

def evaluate_query(rows, score_col):
    # Break score ties consistently for the top-three list.
    ranked = rows.sort_values(
        [score_col, "distance_km", "facility_id"],
        ascending=[False, True, True],
    )
    return {
        "ndcg_at_5": ndcg_score(
            [rows["label"].to_numpy()],
            [rows[score_col].to_numpy()],
            k=5,
        ),
        "precision_at_3": ranked.head(3)["label"].ge(2).mean(),
    }

results = []
df["ridge_test_score"] = np.nan

for train_idx, test_idx in LeaveOneGroupOut().split(X, y, groups):
    model = new_model()
    model.fit(X.iloc[train_idx], y.iloc[train_idx])

    test = df.iloc[test_idx].copy()
    test["ridge_test_score"] = model.predict(X.iloc[test_idx])
    df.loc[test.index, "ridge_test_score"] = test["ridge_test_score"]

    for name, col in [
        ("Current formula", "baseline_score"),
        ("Ridge", "ridge_test_score"),
    ]:
        results.append({
            "query_id": test["query_id"].iloc[0],
            "model": name,
            **evaluate_query(test, col),
        })

assert df["ridge_test_score"].notna().all()
per_query = pd.DataFrame(results)
summary = per_query.groupby("model")[["ndcg_at_5", "precision_at_3"]].mean()

print("Average across 15 held-out queries")
display(summary.round(3).reset_index())

# Count how often Ridge improves, ties, or worsens the top-five ranking.
wide = per_query.pivot(index="query_id", columns="model", values="ndcg_at_5")
delta = (wide["Ridge"] - wide["Current formula"]).round(6)
print(f"Ridge wins: {(delta > 0).sum()}, ties: {(delta == 0).sum()}, "
      f"losses: {(delta < 0).sum()}")

In [0]:
final_model = new_model()
final_model.fit(X, y)

ridge = final_model.named_steps["ridge"]
weights = pd.DataFrame({
    "feature": feature_cols,
    "standardized_weight": ridge.coef_,
})
weights["absolute_weight"] = weights["standardized_weight"].abs()
weights = weights.sort_values("absolute_weight", ascending=False)

print(f"Final model trained on {len(df)} labeled candidates")
print(f"Intercept: {ridge.intercept_:.3f}")
display(weights[["feature", "standardized_weight"]].round(3))